# 🚀 Biến Google Colab Thành Server Thật Với Tên Miền Riêng (Cloudflare Named Tunnel)
Notebook này biến Colab thành một **máy chủ công khai (Public Server)** chuẩn 100% như VPS trên AWS/DigitalOcean:
- **Tên Miền Riêng Cố Định**: Chạy web/API qua domain riêng của bạn (ví dụ `https://gpu.yourdomain.com`). Bất kỳ ai trên thế giới cũng truy cập được, có sẵn chứng chỉ HTTPS bảo mật miễn phí.
- **OpenSSH Server & Tailscale**: Quản trị máy chủ từ xa an toàn qua SSH.
- **Docker & GitHub Actions Runner**: Sẵn sàng phục vụ quy trình CI/CD tự động.

### Bước 1: Giữ phiên làm việc (Keep-Alive JavaScript)
Chạy cell dưới đây để bật script JavaScript giữ kết nối trình duyệt tránh bị Colab ngắt khi chuyển tab.

In [ ]:
from IPython.display import display, Javascript

def keep_alive_js():
    js_code = '''
    function ClickConnect(){
        console.log("[Keep-Alive] Pinging Colab connection...");
        var btn = document.querySelector("colab-connect-button");
        if (btn) btn.click();
    }
    setInterval(ClickConnect, 60000);
    console.log("[Keep-Alive] Script đã kích hoạt (chu kỳ 60s)");
    '''
    display(Javascript(js_code))

keep_alive_js()

### Bước 2: Tải và Cập nhật Mã Nguồn `Colab2Server`

In [ ]:
import os
WORKSPACE_DIR = "/content/Colab2Server"
REPO_URL = "https://github.com/giangpvg/Colab2Server.git"

if not os.path.exists(WORKSPACE_DIR):
    print(">>> Đang tải mã nguồn từ GitHub...")
    !git clone {REPO_URL} {WORKSPACE_DIR}
else:
    print(">>> Đang cập nhật mã nguồn mới nhất...")
    !git -C {WORKSPACE_DIR} pull origin main 2>/dev/null || true

%cd {WORKSPACE_DIR}
!chmod +x scripts/*.sh scripts/*.py 2>/dev/null || true
print("\n>>> Danh sách script sẵn sàng:")
!ls -la scripts/

### Bước 3: Cấu Hình Thông Số (Điền Form Bên Phải ➡️)

#### 📖 Hướng Dẫn Lấy Cloudflare Tunnel Token (Chỉ làm 1 lần duy nhất):
1. Đăng nhập trang quản trị Cloudflare Zero Trust: [one.dash.cloudflare.com](https://one.dash.cloudflare.com/)
2. Vào mục **Networks** > **Tunnels** > Bấm **Create a tunnel** (chọn `Cloudflared`).
3. Đặt tên Tunnel (ví dụ: `colab-server`) > Bấm **Save tunnel**.
4. Ở màn hình Install, nhìn vào dòng lệnh Linux, bạn sẽ thấy chuỗi token sau `--token eyJh...`. **Copy chuỗi token đó** và dán vào ô `CLOUDFLARE_TUNNEL_TOKEN` bên dưới.
5. Chuyển sang tab **Public Hostname** trên Cloudflare:
   - **Subdomain**: ví dụ `gpu` (hoặc để trống nếu dùng domain chính).
   - **Domain**: chọn tên miền riêng của bạn (ví dụ `yourdomain.com`).
   - **Service**: Chọn `HTTP` và URL là `localhost:8000` (cổng của ứng dụng web/API).

In [ ]:
#@title ⚙️ Bảng Cấu Hình Dịch Vụ & Tên Miền Riêng { form-width: "65%" }

# --- 1. GẮN TÊN MIỀN RIÊNG CỐ ĐỊNH (Cloudflare Named Tunnel) ---
ENABLE_CUSTOM_DOMAIN = True #@param {type:"boolean"}
CLOUDFLARE_TUNNEL_TOKEN = "" #@param {type:"string"}
YOUR_CUSTOM_DOMAIN = "gpu.yourdomain.com" #@param {type:"string"}

# --- 2. MẠNG QUẢN TRỊ NỘI BỘ (Tailscale VPN cho SSH) ---
ENABLE_TAILSCALE = True #@param {type:"boolean"}
TAILSCALE_AUTH_KEY = "" #@param {type:"string"}

# --- 3. GÓI CD: Docker & OpenSSH Server ---
ENABLE_DOCKER = True #@param {type:"boolean"}
ENABLE_SSH = True #@param {type:"boolean"}
SSH_ROOT_PASSWORD = "123456" #@param {type:"string"}

# --- 4. GÓI CI: GitHub Actions Runner (Tùy chọn) ---
ENABLE_RUNNER = False #@param {type:"boolean"}
GITHUB_REPO_URL = "https://github.com/giangpvg/Colab2Server" #@param {type:"string"}
RUNNER_TOKEN = "" #@param {type:"string"}
RUNNER_NAME = "colab-runner-01" #@param {type:"string"}
RUNNER_LABELS = "colab,gpu,cuda,staging" #@param {type:"string"}

print("Đã lưu toàn bộ thông số cấu hình!")

### Bước 4: Kích Hoạt Toàn Bộ Dịch Vụ (1-Click Run)
Nhấn nút Play để tự động kết nối Tên miền riêng, OpenSSH, Tailscale và Docker!

In [ ]:
# 1. Khởi tạo hệ thống cơ bản
!bash scripts/setup_system.sh

# 2. Kích hoạt Docker Engine
if ENABLE_DOCKER:
    !bash scripts/setup_docker.sh

# 3. Kích hoạt OpenSSH Server (Lắng nghe tại 0.0.0.0:22 & cấu hình GPU)
if ENABLE_SSH:
    !bash scripts/setup_ssh.sh "{SSH_ROOT_PASSWORD}"

# 4. Kích hoạt Tên Miền Riêng Cố Định qua Cloudflare Named Tunnel
if ENABLE_CUSTOM_DOMAIN and CLOUDFLARE_TUNNEL_TOKEN:
    !bash scripts/setup_tunnel.sh "cloudflare-token" "{CLOUDFLARE_TUNNEL_TOKEN}"
    print("\n==========================================================")
    print("🎉 SERVER ĐÃ KẾT NỐI VỚI TÊN MIỀN RIÊNG CỐ ĐỊNH THÀNH CÔNG!")
    print(f"👉 Bất kỳ ai trên Internet đều có thể truy cập tại: https://{YOUR_CUSTOM_DOMAIN}")
    print("==========================================================\n")
elif ENABLE_CUSTOM_DOMAIN and not CLOUDFLARE_TUNNEL_TOKEN:
    print("⚠️ CẢNH BÁO: Bạn đã bật ENABLE_CUSTOM_DOMAIN nhưng chưa điền CLOUDFLARE_TUNNEL_TOKEN.")

# 5. Kích hoạt Mạng Riêng Ảo Tailscale (nếu có Auth Key)
if ENABLE_TAILSCALE and TAILSCALE_AUTH_KEY:
    !bash scripts/setup_tunnel.sh "tailscale" "{TAILSCALE_AUTH_KEY}"

# 6. Kích hoạt GitHub Actions Runner (nếu có Token)
if ENABLE_RUNNER and RUNNER_TOKEN:
    !bash scripts/setup_runner.sh "{GITHUB_REPO_URL}" "{RUNNER_TOKEN}" "{RUNNER_NAME}" "{RUNNER_LABELS}"
else:
    print("ℹ️ Bỏ qua kích hoạt Runner (RUNNER_TOKEN trống).")

### Bước 5: Giám Sát Tài Nguyên & Duy Trì Phiên (Dashboard)
Chạy cell này để duy trì phiên làm việc không bị ngắt kết nối và theo dõi nhiệt độ, VRAM GPU Tesla T4, RAM và CPU.

In [ ]:
!python3 scripts/keep_alive.py